In [ ]:
# Notebook 01: L1 Dummy Data Generation
# Produces ../data/L1_rolling_updated.csv and ../data/L1_rolling_updated.json - 359 rows, Jan-04 to Dec-28
 
# Cell 1: Imports and global constants
# All window, boundary, and closure settings live here - change here only

import pandas as pd
import numpy as np
import json
from scipy.stats import truncnorm

In [ ]:
# Cell 2: Date spine and seasonal signal
# Phase offset places dummy peak in spring - distinct from real December peak

START      = "2023-01-01"
END        = "2023-12-31"
BIN_ORIGIN = pd.Timestamp("2023-01-01")  # Jan-01 anchor for all 7-day bins
N_BINS     = 52                           # 52 complete bins Jan-01 to Dec-25
ZERO_DATES = ["2023-12-25"]              # set to zero before binning
NULL_DATES = []                           # set to NaN before binning
SEED = 77

print("Constants loaded")
print(f"Bin origin: {BIN_ORIGIN.date()}, bins: {N_BINS}")
print(f"Output range: Jan-01 to Dec-25 ({N_BINS} weekly bins)")

dates        = pd.date_range(START, END, freq="D")
N            = len(dates)
day_idx      = np.arange(N)
 
seasonal     = 1.0 + 0.18 * np.sin((day_idx / N) * 2 * np.pi + np.pi * 0.3)
is_weekend   = pd.Series(dates).dt.dayofweek.isin([5, 6]).astype(float).values
weekend_bump = 1.0 + 0.08 * is_weekend
 
print(f"Date spine: {dates[0].date()} to {dates[-1].date()}, {N} days")

In [ ]:
# Cell 3: Generate raw daily values for all metrics
# Truncated normal keeps each column within realistic physical bounds

def trunc(mean, std, low, high, size):
    a = (low  - mean) / std
    b = (high - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size, random_state=SEED)
 
total_GHGE_SF = trunc(820_000,      95_000,     500_000,    1_100_000,    N) * seasonal * weekend_bump
total_LU_SF   = trunc(2_100_000,   480_000,     900_000,    3_400_000,    N) * seasonal * weekend_bump
total_WU_SF   = trunc(140_000_000, 28_000_000,  60_000_000, 200_000_000,  N) * seasonal * weekend_bump
 
avg_GHGE_perkg = trunc(5.2,   1.1,  2.0,  9.0,  N)
avg_LU_perkg   = trunc(11.8,  2.6,  4.0,  22.0, N)
avg_WU_perkg   = trunc(270.0, 55.0, 80.0, 500.0, N)
 
GHGE_MAX = 95.0
LU_MAX   = 380.0
WU_MAX   = 5_500.0
 
total_GHGE_scaled = total_GHGE_SF / GHGE_MAX
total_LU_scaled   = total_LU_SF   / LU_MAX
total_WU_scaled   = total_WU_SF   / WU_MAX
 
avg_composite_SF             = (total_GHGE_scaled + total_LU_scaled + total_WU_scaled) / 3.0
avg_composite_SF_norm        = avg_composite_SF / avg_composite_SF.max()
 
GHGE_perkg_norm              = avg_GHGE_perkg / avg_GHGE_perkg.max()
LU_perkg_norm                = avg_LU_perkg   / avg_LU_perkg.max()
WU_perkg_norm                = avg_WU_perkg   / avg_WU_perkg.max()
avg_composite_intensity_norm = (GHGE_perkg_norm + LU_perkg_norm + WU_perkg_norm) / 3.0
 
for name, arr in [
    ("total_GHGE_SF",                total_GHGE_SF),
    ("total_LU_SF",                  total_LU_SF),
    ("total_WU_SF",                  total_WU_SF),
    ("avg_GHGE_perkg",               avg_GHGE_perkg),
    ("avg_composite_SF_norm",        avg_composite_SF_norm),
    ("avg_composite_intensity_norm", avg_composite_intensity_norm),
]:
    print(f"  {name:<35} min={arr.min():.4f}  max={arr.max():.4f}")

In [ ]:
# Cell 4: Build raw L1 DataFrame and apply closure masks
# Masks applied before rolling so closure days contribute correctly to windows

ALL_SRC_COLS = [
    "total_GHGE_SF", "total_LU_SF", "total_WU_SF",
    "avg_GHGE_perkg", "avg_LU_perkg", "avg_WU_perkg",
    "avg_composite_SF_norm", "avg_composite_intensity_norm",
]
 
L1 = pd.DataFrame({
    "date"                         : [d.strftime("%Y-%m-%d") for d in dates],
    "total_GHGE_SF"                : total_GHGE_SF.round(2),
    "total_LU_SF"                  : total_LU_SF.round(2),
    "total_WU_SF"                  : total_WU_SF.round(2),
    "avg_GHGE_perkg"               : avg_GHGE_perkg.round(6),
    "avg_LU_perkg"                 : avg_LU_perkg.round(6),
    "avg_WU_perkg"                 : avg_WU_perkg.round(6),
    "avg_composite_SF_norm"        : avg_composite_SF_norm.round(6),
    "avg_composite_intensity_norm" : avg_composite_intensity_norm.round(6),
})
 
for d in ZERO_DATES:
    mask = L1["date"] == d
    L1.loc[mask, ALL_SRC_COLS] = 0.0
    print(f"Zero mask applied: {d} ({int(mask.sum())} rows)")
 
for d in NULL_DATES:
    mask = L1["date"] == d
    L1.loc[mask, ALL_SRC_COLS] = np.nan
    print(f"Null mask applied: {d} ({int(mask.sum())} rows)")
 
print(f"L1 shape: {L1.shape}")

In [ ]:
# ==============================================================================
# Cell 5: Aggregate daily values into 52 fixed 7-day bins
# Jan-01 anchored: bin 1 = Jan-01 to Jan-07, bin 2 = Jan-08 to Jan-14, etc.
# No boundary exclusion needed -- every bin has exactly 7 days.
# ZERO_DATES contribute zero to their bin mean (correct -- real closure day).
# NULL_DATES excluded via NaN before binning -- not used in dummy data.
# Column names use roll7_ prefix for chart backward compatibility.
# ==============================================================================

# Map source column names to output column names.
# Values are 7-day bin means not rolling means -- naming kept for chart compat.
BIN_MAP = {
    "total_GHGE_SF"                : "roll7_GHGE_SF",
    "total_LU_SF"                  : "roll7_LU_SF",
    "total_WU_SF"                  : "roll7_WU_SF",
    "avg_composite_SF_norm"        : "roll7_composite_norm",
    "avg_GHGE_perkg"               : "roll7_GHGE_perkg",
    "avg_LU_perkg"                 : "roll7_LU_perkg",
    "avg_WU_perkg"                 : "roll7_WU_perkg",
    "avg_composite_intensity_norm" : "roll7_composite_intensity_norm",
}

# Convert date string to datetime for resample -- operates on index.
L1_dt = L1.copy()
L1_dt["date"] = pd.to_datetime(L1_dt["date"])
L1_dt = L1_dt.set_index("date")

# Resample to 7-day bins anchored at Jan-01.
# Each bin label is the first date of that bin.
L1_binned = (
    L1_dt[list(BIN_MAP.keys())]
    .resample("7D", origin=BIN_ORIGIN)
    .mean()
    .reset_index()
)

# Keep exactly N_BINS complete bins -- drops the partial Dec-26 to Dec-31 bin.
L1_binned = L1_binned.head(N_BINS).copy()

# Rename source columns to output names and format date as string.
L1_binned = L1_binned.rename(columns=BIN_MAP)
L1_binned["week_start"] = L1_binned["date"].dt.strftime("%Y-%m-%d")
L1_binned = L1_binned.drop(columns=["date"])

# Round all value columns to 6 decimal places.
for col in BIN_MAP.values():
    L1_binned[col] = L1_binned[col].round(6)

# Verification: all bins present, no nulls, correct date range.
print(f"Bin count: {len(L1_binned)} (expected {N_BINS})")
print(f"First bin: {L1_binned['week_start'].iloc[0]}")
print(f"Last bin:  {L1_binned['week_start'].iloc[-1]}")
ref_n = len(L1_binned)
for col in BIN_MAP.values():
    n    = L1_binned[col].notna().sum()
    flag = "OK" if n == ref_n else "MISMATCH"
    print(f"  {col:<40} n={n}  {flag}")

In [ ]:
EXPORT_COLS = [
    "week_start",
    "roll7_GHGE_SF",
    "roll7_LU_SF",
    "roll7_WU_SF",
    "roll7_composite_norm",
    "roll7_GHGE_perkg",
    "roll7_LU_perkg",
    "roll7_WU_perkg",
    "roll7_composite_intensity_norm",
]

L1_export = L1_binned[EXPORT_COLS].copy()

# Null check: bin means should never be null -- complete 7-day windows.
null_counts = L1_export.drop(columns=["week_start"]).isnull().sum()
print("Null counts (all must be 0):")
for col, n in null_counts.items():
    print(f"  {col:<40} {n}")

print(f"Rows: {len(L1_export)} (expected {N_BINS})")
print(f"Range: {L1_export['week_start'].iloc[0]} -> {L1_export['week_start'].iloc[-1]}")

L1_export.to_csv("../data/L1_rolling_updated.csv", index=False)
print("Saved: ../data/L1_rolling_updated.csv")

L1_export.to_json("../data/L1_rolling_updated.json", orient="records", indent=2)
print("Saved: ../data/L1_rolling_updated.json")

with open("../data/L1_rolling_updated.json") as f:
    check = json.load(f)
print(f"JSON records: {len(check)}")
print(f"First: {check[0]['week_start']}  roll7_GHGE_SF={check[0]['roll7_GHGE_SF']}")
print(f"Last:  {check[-1]['week_start']}  roll7_GHGE_SF={check[-1]['roll7_GHGE_SF']}")

In [ ]:
# Bin 1 = Jan-01 to Jan-07 -- 7 days.

bin1_val    = L1_export["roll7_GHGE_SF"].iloc[0]
manual_mean = L1[L1["date"] <= "2023-01-07"]["total_GHGE_SF"].mean()
diff        = abs(bin1_val - manual_mean)

print(f"Bin 1 GHGE SF (exported):    {bin1_val:.4f}")
print(f"Bin 1 GHGE SF (manual mean): {manual_mean:.4f}")
print(f"Difference:                  {diff:.6f}  {'OK' if diff < 0.01 else 'MISMATCH'}")

In [ ]:
# P2 spot check: bin 52 (Dec-19 to Dec-25) must be non-null.
# Christmas Day (Dec-25) zero pulls the bin mean slightly below surrounding weeks.

bin52 = L1_export["roll7_GHGE_SF"].iloc[51]
bin51 = L1_export["roll7_GHGE_SF"].iloc[50]

print(f"Bin 51 GHGE SF (Dec-12 to Dec-18): {bin51:.2f}")
print(f"Bin 52 GHGE SF (Dec-19 to Dec-25): {bin52:.2f}")
print(f"Bin 52 lower than bin 51: {'YES -- Christmas zero visible' if bin52 < bin51 else 'NO -- check ZERO_DATES mask'}")
print(f"Bin 52 non-null: {'OK' if not pd.isna(bin52) else 'MISMATCH'}")

In [ ]:
# P2 null sweep: all 8 value columns across all 52 rows must be non-null.

null_total = L1_export.drop(columns=["week_start"]).isnull().sum().sum()
print(f"Total nulls across all value columns: {null_total}  {'OK' if null_total == 0 else 'MISMATCH'}")